# 🔱 Shiv AI Voice Cloning v4.0
**Owner: Shri Ram Nag | PAISAWALA Channel**

## 🆕 v4 Updates:
| Fix/Feature | Detail |
|---|---|
| ✅ **Voice Clone Bug Fix** | Reference audio normalization — quiet audio bhi kaam karega |
| ✅ **Audio Pre-processing** | Auto-denoise, normalize, resample before cloning |
| ✅ **Clone Quality Boost** | guidance_scale 3.5, steps 40 — better similarity |
| ✅ **Long Script Fix** | Smart chunking v2 — ellipsis preserve, no dropout |
| 🆕 **Emotion Control** | Happy/Sad/Angry/Fear/Disgust/Neutral presets |
| 🆕 **Audio Download** | Direct WAV download button |
| 🆕 **Clone + Design** | Reference audio se clone + speed/pitch apply |
| 🆕 **Debug Mode** | Errors clearly dikhenge |

> ⚡ **PEHLE: Runtime → Change runtime type → T4 GPU select karein!**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1: GPU Check
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
print('=' * 50)
print('🔱 SHIV AI VOICE CLONING v4.0')
print('   Owner: Shri Ram Nag | PAISAWALA')
print('=' * 50)
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {round(torch.cuda.get_device_properties(0).total_memory/1e9,2)} GB')
    print('✅ GPU Ready!')
else:
    raise RuntimeError('❌ GPU nahi mila! Runtime → Change runtime type → T4 GPU!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2: Install All Dependencies
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('📦 Installing dependencies...')
!pip install -q gradio==4.44.0
!pip install -q huggingface_hub transformers accelerate
!pip install -q scipy numpy librosa soundfile
!pip install -q noisereduce
print('✅ All dependencies installed!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3: Download Model from HuggingFace
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os
from huggingface_hub import snapshot_download

REPO_ID   = 'Shriramnag/Shiv-AI-Voice-Cloning'
LOCAL_DIR = './Shiv-AI-Voice-Cloning'

if not os.path.exists(LOCAL_DIR) or len(os.listdir(LOCAL_DIR)) < 5:
    print(f'📥 Downloading {REPO_ID}...')
    print('⏳ ~3.27 GB — please wait (5-10 min on Colab free tier)')
    snapshot_download(
        repo_id=REPO_ID,
        local_dir=LOCAL_DIR,
        local_dir_use_symlinks=False,
        ignore_patterns=['*.md','*.txt'],  # model files only
    )
    print('✅ Download complete!')
else:
    print('✅ Model already present locally!')

print('\n📂 Downloaded files:')
for f in sorted(os.listdir(LOCAL_DIR)):
    path = os.path.join(LOCAL_DIR, f)
    if os.path.isfile(path):
        sz = os.path.getsize(path) / 1e6
        print(f'  {f:45s} {sz:8.2f} MB')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 4: Import & Load Model
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import sys, re, io, tempfile, logging
import numpy as np
import torch
import scipy.io.wavfile as wavfile
import librosa
import soundfile as sf
import gradio as gr

try:
    import noisereduce as nr
    HAS_NR = True
except ImportError:
    HAS_NR = False
    print('Note: noisereduce not available, skipping denoise step')

MODEL_PATH = './Shiv-AI-Voice-Cloning'
sys.path.insert(0, MODEL_PATH)

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name
try:
    from subtitle import LANGUAGE_CODE as WHISPER_LANGUAGE_CODE
except ImportError:
    WHISPER_LANGUAGE_CODE = None

logging.basicConfig(level=logging.WARNING)

print('🔱 Loading Shiv AI Voice Cloning model...')
model = OmniVoice.from_pretrained(
    MODEL_PATH,
    device_map='cuda',
    dtype=torch.float16,
    load_asr=False,
)
SR = model.sampling_rate
print(f'✅ Model loaded! Sampling Rate = {SR} Hz')
os.makedirs('./Shiv_Audio', exist_ok=True)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 5: Core Utility Functions
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ─── 🔧 FIX #1: Audio Pre-processor (Main Bug Fix) ───────────────────────
def preprocess_ref_audio(audio_path, target_sr=24000):
    """
    ✅ Voice Clone ka main bug yahan tha:
    - Reference audio quiet tha (amplitude 423 / 32767 = 1.3% only!)
    - Model ko proper loudness chahiye clone karne ke liye

    Kya karta hai:
    1. Audio load karo (any format/sample_rate)
    2. Mono convert karo
    3. Target sample rate pe resample (24000 Hz)
    4. Normalize karo -3 dBFS tak (model ke liye optimal)
    5. Optional: noise reduce
    6. Processed audio temp file mein save karo
    """
    try:
        # Load audio (any format)
        audio, orig_sr = librosa.load(audio_path, sr=None, mono=True)
        print(f'  📊 Original: SR={orig_sr}Hz | Duration={len(audio)/orig_sr:.2f}s | Peak={np.abs(audio).max():.4f}')

        # Resample if needed
        if orig_sr != target_sr:
            audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=target_sr)
            print(f'  🔄 Resampled: {orig_sr}Hz → {target_sr}Hz')

        # Noise reduce (if available)
        if HAS_NR and np.abs(audio).max() > 0.001:
            audio = nr.reduce_noise(y=audio, sr=target_sr, stationary=False, prop_decrease=0.7)
            print(f'  🔇 Noise reduced')

        # ✅ KEY FIX: Normalize to -3 dBFS
        peak = np.abs(audio).max()
        if peak < 0.01:
            raise ValueError(f'Audio bahut quiet hai (peak={peak:.4f}). Louder audio record karein!')
        target_peak = 10 ** (-3.0 / 20)  # -3 dBFS = 0.7079
        audio = audio * (target_peak / peak)
        print(f'  🔊 Normalized: peak {peak:.4f} → {np.abs(audio).max():.4f}')

        # Trim silence from edges
        audio, _ = librosa.effects.trim(audio, top_db=20)
        print(f'  ✂️  Trimmed: {len(audio)/target_sr:.2f}s')

        # Save to temp file
        tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False, dir='./Shiv_Audio')
        sf.write(tmp.name, audio, target_sr)
        print(f'  💾 Saved: {tmp.name}')
        return tmp.name

    except Exception as e:
        print(f'  ⚠️ Preprocess warning: {e} — using original')
        return audio_path

# ─── 🔧 FIX #2: Smart Chunking ───────────────────────────────────────────
def split_chunks(text, max_ch=90):
    """
    v2 Fix: Newline-based merging, ellipsis preserve, no chunk dropout
    """
    raw_lines = text.split('\n')
    lines = [l.strip() for l in raw_lines if l.strip()]
    chunks, cur = [], ''
    for line in lines:
        if len(line) > max_ch:
            if cur:
                chunks.append(cur)
                cur = ''
            parts = re.split(r'(?<=[।.!?,…])\s*', line)
            sub = ''
            for p in parts:
                p = p.strip()
                if not p: continue
                if len(sub) + len(p) + 1 <= max_ch:
                    sub = (sub + ' ' + p).strip() if sub else p
                else:
                    if sub: chunks.append(sub)
                    sub = p
            if sub: chunks.append(sub)
        else:
            merged = (cur + ' ' + line).strip() if cur else line
            if len(merged) <= max_ch:
                cur = merged
            else:
                if cur: chunks.append(cur)
                cur = line
    if cur: chunks.append(cur)
    return [c for c in chunks if c.strip()]

# ─── Audio Utilities ──────────────────────────────────────────────────────
def join_audio(audios, sil_ms=0):
    if not audios: return np.zeros(1, dtype=np.float32)
    if sil_ms > 0:
        sil = np.zeros(int(SR * sil_ms / 1000), dtype=np.float32)
        out = []
        for i, a in enumerate(audios):
            out.append(a)
            if i < len(audios) - 1: out.append(sil)
        return np.concatenate(out)
    return np.concatenate(audios)

def to_wav(a):
    a_clipped = np.clip(a, -1.0, 1.0)
    return (SR, (a_clipped * 32767).astype(np.int16))

def make_cfg(steps=40, gs=2.0, speed=1.0, pitch=0, energy=1.0):
    """Generate config — graceful fallback if model doesn't support speed/pitch/energy"""
    try:
        return OmniVoiceGenerationConfig(
            num_step=steps, guidance_scale=gs,
            denoise=True, preprocess_prompt=True, postprocess_output=True,
            speed=speed, pitch=pitch, energy=energy
        )
    except TypeError:
        return OmniVoiceGenerationConfig(
            num_step=steps, guidance_scale=gs,
            denoise=True, preprocess_prompt=True, postprocess_output=True
        )

def run_chunk(text, lang, cfg, vcp=None, inst=None):
    kw = dict(
        text=text,
        language=lang if lang != 'Auto' else None,
        generation_config=cfg
    )
    if vcp:  kw['voice_clone_prompt'] = vcp
    if inst: kw['instruct'] = inst
    out = model.generate(**kw)
    return out[0]

print('✅ Utility functions loaded!')

# Quick test
t = split_chunks('रुकिए…\nएक पल के लिए।\nयह एक गहरी बात है…')
print(f'🧪 Chunking test: {len(t)} chunks — {t}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 6: Tab Generation Functions
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def fn_clone(text, lang, ref, ref_text, clone_steps, clone_gs):
    """Tab 1: Voice Clone with audio preprocessing fix"""
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    if not ref:                      return None, '⚠️ Reference audio upload karein'
    try:
        print(f'🎙️ Voice Clone started...')
        print(f'  Text length: {len(text)} chars')

        # ✅ KEY FIX: Preprocess reference audio
        print('  🔧 Preprocessing reference audio...')
        processed_ref = preprocess_ref_audio(ref, target_sr=SR)

        # Create voice clone prompt
        vcp = model.create_voice_clone_prompt(
            ref_audio=processed_ref,
            ref_text=ref_text.strip() if ref_text and ref_text.strip() else None
        )
        print(f'  ✅ VCP created!')

        # Chunk & generate
        chunks = split_chunks(text)
        print(f'  📦 {len(chunks)} chunks: {[c[:30]+"..." for c in chunks[:3]]}')

        cfg = make_cfg(steps=int(clone_steps), gs=float(clone_gs))
        audio_parts = []
        for i, c in enumerate(chunks):
            print(f'  🔊 Chunk {i+1}/{len(chunks)}: "{c[:40]}"')
            audio_parts.append(run_chunk(c, lang, cfg, vcp=vcp))

        audio = join_audio(audio_parts)
        dur = len(audio) / SR
        print(f'  ✅ Generated {dur:.1f}s audio!')
        return to_wav(audio), f'✅ Clone done! {len(chunks)} chunks | {dur:.1f}s audio'
    except Exception as e:
        import traceback
        err = traceback.format_exc()
        print(f'❌ Error:\n{err}')
        return None, f'❌ Error: {str(e)}'


def fn_clone_design(text, lang, ref, ref_text, speed, pitch, energy, pause_ms):
    """Tab 2: Clone + Voice Design combined"""
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    if not ref:                      return None, '⚠️ Reference audio upload karein'
    try:
        print(f'🎛️ Clone+Design: speed={speed} pitch={pitch} energy={energy}')
        processed_ref = preprocess_ref_audio(ref, target_sr=SR)
        vcp = model.create_voice_clone_prompt(
            ref_audio=processed_ref,
            ref_text=ref_text.strip() if ref_text and ref_text.strip() else None
        )
        cfg = make_cfg(steps=40, gs=3.0, speed=speed, pitch=pitch, energy=energy)
        chunks = split_chunks(text)
        audio = join_audio(
            [run_chunk(c, lang, cfg, vcp=vcp) for c in chunks],
            sil_ms=int(pause_ms)
        )
        return to_wav(audio), f'✅ Clone+Design! speed={speed} pitch={pitch} | {len(audio)/SR:.1f}s'
    except Exception as e:
        return None, f'❌ {e}'


def fn_design(text, lang, speed, pitch, energy, pause_ms, style, emotion):
    """Tab 3: Voice Design with emotion presets"""
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        # Emotion + style combine karo
        emotion_map = {
            'Neutral':  '',
            'Happy 😊': 'speak with happiness, warmth and positivity',
            'Sad 😢':   'speak in a sad, melancholic and slow tone with pauses',
            'Angry 😠': 'speak with anger, firmness and strong emphasis',
            'Fear 😨':  'speak with fear, nervousness and shaky voice',
            'Excited 🔥': 'speak with high excitement and energy',
            'Calm 😌':  'speak very calmly and peacefully',
            'Mysterious 🌙': 'speak slowly, mysteriously with dramatic pauses',
        }
        emotion_inst = emotion_map.get(emotion, '')
        combined_inst = ' '.join(filter(None, [emotion_inst, style.strip()])).strip() or None

        cfg = make_cfg(steps=36, gs=2.5, speed=speed, pitch=pitch, energy=energy)
        chunks = split_chunks(text)
        audio = join_audio(
            [run_chunk(c, lang, cfg, inst=combined_inst) for c in chunks],
            sil_ms=int(pause_ms)
        )
        return to_wav(audio), f'✅ Done! emotion={emotion} speed={speed} pitch={pitch} | {len(audio)/SR:.1f}s'
    except Exception as e:
        return None, f'❌ {e}'


def fn_tts(text, lang, steps, gs):
    """Tab 4: Simple TTS"""
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        chunks = split_chunks(text)
        audio = join_audio([run_chunk(c, lang, make_cfg(int(steps), float(gs))) for c in chunks])
        return to_wav(audio), f'✅ TTS done! {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e:
        return None, f'❌ {e}'


def fn_instruct(text, lang, prompt):
    """Tab 5: Instruct Mode"""
    if not text   or not text.strip():   return None, '⚠️ Text likhein'
    if not prompt or not prompt.strip(): return None, '⚠️ Instruction likhein'
    try:
        chunks = split_chunks(text)
        audio = join_audio([run_chunk(c, lang, make_cfg(gs=3.0), inst=prompt.strip()) for c in chunks])
        return to_wav(audio), f'✅ Done! {len(audio)/SR:.1f}s'
    except Exception as e:
        return None, f'❌ {e}'


def fn_audio_check(ref):
    """Reference audio quality check"""
    if not ref: return '⚠️ Koi audio upload nahi hua'
    try:
        audio, orig_sr = librosa.load(ref, sr=None, mono=True)
        peak = np.abs(audio).max()
        dur  = len(audio) / orig_sr
        rms  = np.sqrt(np.mean(audio**2))
        db   = 20 * np.log10(peak) if peak > 0 else -99

        status = ''
        if peak < 0.05:
            status = '❌ BAHUT QUIET — Clone kaam nahi karega! Louder audio chahiye'
        elif peak < 0.2:
            status = '⚠️ Thoda quiet — preprocess karke try karenge'
        elif dur < 3:
            status = '⚠️ Bahut chhota audio (< 3 sec) — 5-30 sec best hota hai'
        elif dur > 60:
            status = '⚠️ Bahut lamba audio (> 60 sec) — 5-30 sec use karein'
        else:
            status = '✅ Audio quality GOOD — clone ho sakta hai!'

        return (
            f'📊 Audio Analysis:\n'
            f'  Duration    : {dur:.2f} seconds\n'
            f'  Sample Rate : {orig_sr} Hz\n'
            f'  Peak Level  : {peak:.4f} ({db:.1f} dBFS)\n'
            f'  RMS Level   : {rms:.4f}\n'
            f'  Status      : {status}'
        )
    except Exception as e:
        return f'❌ Audio read error: {e}'


print('✅ All generation functions ready!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 7: UI Constants
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LANG_CHOICES  = ['Auto'] + sorted(lang_display_name(n) for n in LANG_NAMES)
EVENT_TAGS    = ['[laughter]','[sigh]','[confirmation-en]','[question-en]','[surprise-wa]','[dissatisfaction-hnn]']
EMOTIONS      = ['Neutral','Happy 😊','Sad 😢','Angry 😠','Fear 😨','Excited 🔥','Calm 😌','Mysterious 🌙']
INSTRUCT_EX   = [
    'Speak slowly and clearly with a calm, deep voice',
    'Speak with excitement and high energy',
    'Speak softly like a bedtime story narrator',
    'Speak like a professional news anchor, formal and clear',
    'Speak in a sad, emotional tone with natural pauses',
    'Fast and enthusiastic like a radio jockey',
    'Speak mysteriously and dramatically with long pauses',
    'धीरे, शांत और गहरी आवाज़ में बोलें',
    'जोश और उत्साह के साथ तेज़ आवाज़ में बोलें',
    'एक कहानीकार की तरह, गर्मजोशी से बोलें',
]
INSERT_TAG_JS = """
(tag_val, cur) => {
    const ta = document.querySelector('.shiv-tb textarea');
    if (!ta) return cur + ' ' + tag_val;
    const s = ta.selectionStart, e = ta.selectionEnd;
    return cur.slice(0,s) + ' ' + tag_val + ' ' + cur.slice(e);
}
"""

CSS = """
.gradio-container {max-width:100%!important;}
footer {display:none!important;}
.shiv-hdr {text-align:center; padding:28px 20px 20px; background:linear-gradient(135deg,#1a0800,#0d0d0d); border-bottom:2px solid #ff6600;}
.shiv-hdr h1 {font-size:2.4em; color:#ff6600; margin:0 0 6px 0;}
.shiv-hdr p  {color:#aaa; font-size:.9em; margin:3px 0;}
.tag-btn  {background:#fff3e0!important; border:1px solid #ffcc80!important; color:#e65100!important; font-size:.75em!important;}
.fix-badge {background:#1a3a1a; border:1px solid #2d7a2d; border-radius:6px; padding:8px 12px; margin:8px 0; color:#5fba5f; font-size:.85em;}
"""

print('✅ UI constants ready!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 8: Launch Gradio UI
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
theme = gr.themes.Soft(primary_hue='orange', font=['Inter','Arial','sans-serif'])

with gr.Blocks(theme=theme, css=CSS, title='🔱 Shiv AI Voice Cloning v4') as demo:

    gr.HTML("""
        <div class='shiv-hdr'>
            <h1>🔱 Shiv AI Voice Cloning</h1>
            <p><b style='color:#ff8c00;'>Owner: Shri Ram Nag</b> &nbsp;|&nbsp; PAISAWALA 🎬 &nbsp;|&nbsp; v4.0</p>
            <p>Model: Shriramnag/Shiv-AI-Voice-Cloning &nbsp;|&nbsp; 646 Languages &nbsp;|&nbsp; ✅ Clone Bug Fixed</p>
        </div>
    """)

    with gr.Tabs():

        # ═══════════════════════════════════════════════════════
        # TAB 1: VOICE CLONE (Fixed)
        # ═══════════════════════════════════════════════════════
        with gr.TabItem('🎙️ Voice Clone ✅'):
            gr.HTML("<div class='fix-badge'>✅ v4 Fix: Reference audio auto-normalize + denoise — quiet audio bhi kaam karega!</div>")
            with gr.Row():
                with gr.Column(scale=1):
                    vc_text = gr.Textbox(
                        label='📝 Text (puri script paste karein)',
                        lines=9, elem_classes='shiv-tb',
                        placeholder='यहाँ पूरी script paste करें...'
                    )
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b.click(fn=None, inputs=[b,vc_text], outputs=vc_text, js=INSERT_TAG_JS)
                    vc_lang     = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    vc_ref      = gr.Audio(label='🎤 Reference Audio (5-30 sec, clear voice)', type='filepath')
                    vc_ref_text = gr.Textbox(label='📄 Reference Transcript (optional — better similarity ke liye)', lines=2)
                    with gr.Row():
                        vc_check_btn = gr.Button('🔍 Audio Quality Check', size='sm')
                    vc_quality  = gr.Textbox(label='Audio Analysis', interactive=False, lines=6)
                    with gr.Row():
                        vc_steps = gr.Slider(label='⚙️ Steps (quality)', minimum=20, maximum=60, value=40, step=4)
                        vc_gs    = gr.Slider(label='🎯 Guidance', minimum=1.5, maximum=5.0, value=3.5, step=0.5)
                    vc_btn = gr.Button('🔱 Clone Voice', variant='primary', size='lg')
                with gr.Column(scale=1):
                    vc_out    = gr.Audio(label='🔊 Cloned Voice Output', type='numpy')
                    vc_status = gr.Textbox(label='Status / Log', interactive=False, lines=4)
                    gr.Markdown("""
**📌 Tips for best clone quality:**
- **5-30 second** ka clear audio best hota hai
- Background noise bilkul na ho
- Ek hi speaker ki awaaz ho
- Reference transcript likhne se similarity ↑↑
- Steps 40-50 = best quality (slow but worth it)
                    """)

            vc_check_btn.click(fn_audio_check, inputs=[vc_ref], outputs=[vc_quality])
            vc_btn.click(fn_clone, inputs=[vc_text,vc_lang,vc_ref,vc_ref_text,vc_steps,vc_gs], outputs=[vc_out,vc_status])

        # ═══════════════════════════════════════════════════════
        # TAB 2: CLONE + DESIGN
        # ═══════════════════════════════════════════════════════
        with gr.TabItem('🎙️+🎛️ Clone+Design'):
            gr.Markdown('### Clone aawaz + Speed/Pitch/Energy control — ek saath!')
            with gr.Row():
                with gr.Column(scale=1):
                    cd_text = gr.Textbox(label='📝 Text', lines=7, elem_classes='shiv-tb', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b.click(fn=None, inputs=[b,cd_text], outputs=cd_text, js=INSERT_TAG_JS)
                    cd_lang     = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    cd_ref      = gr.Audio(label='🎤 Reference Audio', type='filepath')
                    cd_ref_text = gr.Textbox(label='📄 Reference Transcript (optional)', lines=2)
                    cd_speed    = gr.Slider(label='⚡ Speed',  minimum=0.5, maximum=2.0, value=1.0, step=0.05)
                    cd_pitch    = gr.Slider(label='🎵 Pitch',  minimum=-12, maximum=12,  value=0,   step=1)
                    cd_energy   = gr.Slider(label='💪 Energy', minimum=0.3, maximum=2.0, value=1.0, step=0.05)
                    cd_pause    = gr.Slider(label='⏸️ Pause (ms)', minimum=0, maximum=500, value=0, step=50)
                    cd_btn      = gr.Button('🎙️+🎛️ Clone + Design', variant='primary', size='lg')
                with gr.Column(scale=1):
                    cd_out    = gr.Audio(label='🔊 Output', type='numpy')
                    cd_status = gr.Textbox(label='Status', interactive=False)
            cd_btn.click(fn_clone_design, [cd_text,cd_lang,cd_ref,cd_ref_text,cd_speed,cd_pitch,cd_energy,cd_pause], [cd_out,cd_status])

        # ═══════════════════════════════════════════════════════
        # TAB 3: VOICE DESIGN + EMOTIONS
        # ═══════════════════════════════════════════════════════
        with gr.TabItem('🎛️ Voice Design'):
            with gr.Row():
                with gr.Column(scale=1):
                    vd_text    = gr.Textbox(label='📝 Text', lines=7, elem_classes='shiv-tb', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b2 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b2.click(fn=None, inputs=[b2,vd_text], outputs=vd_text, js=INSERT_TAG_JS)
                    vd_lang    = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    vd_emotion = gr.Radio(label='🎭 Emotion', choices=EMOTIONS, value='Neutral')
                    vd_speed   = gr.Slider(label='⚡ Speed',  minimum=0.5, maximum=2.0, value=1.0, step=0.05, info='0.5=Slow | 1.0=Normal | 2.0=Fast')
                    vd_pitch   = gr.Slider(label='🎵 Pitch',  minimum=-12, maximum=12,  value=0,   step=1,    info='-12=Deep | 0=Normal | +12=High')
                    vd_energy  = gr.Slider(label='💪 Energy', minimum=0.3, maximum=2.0, value=1.0, step=0.05, info='0.3=Soft | 1.0=Normal | 2.0=Loud')
                    vd_pause   = gr.Slider(label='⏸️ Pause (ms)', minimum=0, maximum=500, value=0, step=50)
                    vd_style   = gr.Textbox(label='✍️ Extra Style Instruction', lines=2, placeholder='optional...')
                    with gr.Row():
                        pc = gr.Button('😌 Calm',          size='sm')
                        pe = gr.Button('🔥 Excited',       size='sm')
                        pn = gr.Button('📺 News Anchor',   size='sm')
                        ps = gr.Button('📖 Story',         size='sm')
                        pm = gr.Button('🌙 Mysterious',    size='sm')
                    vd_btn = gr.Button('🎛️ Design & Generate', variant='primary', size='lg')
                with gr.Column(scale=1):
                    vd_out    = gr.Audio(label='🔊 Voice Design Output', type='numpy')
                    vd_status = gr.Textbox(label='Status', interactive=False)
                    gr.Markdown("""
| Control | Effect |
|---------|--------|
| Speed ↓ | Dheere bolta hai |
| Speed ↑ | Tez bolta hai |
| Pitch ↓ | Moti/deep awaaz |
| Pitch ↑ | Patli/high awaaz |
| Energy ↓ | Soft/whisper |
| Energy ↑ | Bold/loud |
                    """)

            pc.click(fn=lambda:(0.80, -2, 0.70, 150, 'speak calmly and peacefully'),                          outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            pe.click(fn=lambda:(1.30,  3, 1.50,   0, 'speak with excitement and high energy'),               outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            pn.click(fn=lambda:(1.00,  0, 1.10, 200, 'speak like a professional news anchor, formal clear'), outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            ps.click(fn=lambda:(0.85, -1, 0.80, 250, 'speak like a storyteller, warm and engaging'),        outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            pm.click(fn=lambda:(0.75, -2, 0.75, 350, 'speak mysteriously and dramatically'),                outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            vd_btn.click(fn_design, [vd_text,vd_lang,vd_speed,vd_pitch,vd_energy,vd_pause,vd_style,vd_emotion], [vd_out,vd_status])

        # ═══════════════════════════════════════════════════════
        # TAB 4: SIMPLE TTS
        # ═══════════════════════════════════════════════════════
        with gr.TabItem('🔤 Simple TTS'):
            with gr.Row():
                with gr.Column(scale=1):
                    tts_text = gr.Textbox(label='📝 Text', lines=8, elem_classes='shiv-tb', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b3 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b3.click(fn=None, inputs=[b3,tts_text], outputs=tts_text, js=INSERT_TAG_JS)
                    tts_lang     = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    tts_steps    = gr.Slider(label='🔢 Steps (Quality)', minimum=10, maximum=64, value=32, step=2, info='Zyada = better but slower')
                    tts_guidance = gr.Slider(label='🎯 Guidance Scale',  minimum=1.0, maximum=5.0, value=2.0, step=0.5)
                    tts_btn      = gr.Button('🔤 Generate TTS', variant='primary', size='lg')
                with gr.Column(scale=1):
                    tts_out    = gr.Audio(label='🔊 TTS Output', type='numpy')
                    tts_status = gr.Textbox(label='Status', interactive=False)
            tts_btn.click(fn_tts, [tts_text,tts_lang,tts_steps,tts_guidance], [tts_out,tts_status])

        # ═══════════════════════════════════════════════════════
        # TAB 5: INSTRUCT MODE
        # ═══════════════════════════════════════════════════════
        with gr.TabItem('📋 Instruct Mode'):
            with gr.Row():
                with gr.Column(scale=1):
                    inst_text   = gr.Textbox(label='📝 Text', lines=6, elem_classes='shiv-tb', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b4 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b4.click(fn=None, inputs=[b4,inst_text], outputs=inst_text, js=INSERT_TAG_JS)
                    inst_lang   = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    inst_prompt = gr.Textbox(label='📋 Style Instruction', lines=3,
                                            placeholder='Speak slowly and clearly with a deep male voice...')
                    gr.Markdown('**💡 Examples (click karein):**')
                    for ex in INSTRUCT_EX:
                        eb = gr.Button(ex, size='sm')
                        eb.click(fn=lambda x=ex: x, outputs=inst_prompt)
                    inst_btn = gr.Button('📋 Generate with Instruct', variant='primary', size='lg')
                with gr.Column(scale=1):
                    inst_out    = gr.Audio(label='🔊 Instruct Output', type='numpy')
                    inst_status = gr.Textbox(label='Status', interactive=False)
            inst_btn.click(fn_instruct, [inst_text,inst_lang,inst_prompt], [inst_out,inst_status])

    gr.HTML("<div style='text-align:center;padding:15px;color:#666;border-top:1px solid #333;margin-top:20px;'>© 2026 🔱 Shiv AI Voice Cloning v4.0 &nbsp;|&nbsp; Shri Ram Nag &nbsp;|&nbsp; PAISAWALA</div>")

print('🚀 Launching Shiv AI v4...')
demo.launch(share=True, debug=True)
print('🔱 Shiv AI v4 is LIVE!')